# qust indicator examples

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


本节 用 GitHub K 线数据展示 qust 的指标计算、形态检测和 monitor 绘图。核心计算尽量放在 qust 表达式里；Python 只负责路径、样本大小、展示行数这类边界控制。

已覆盖：

- Hurst exponent；
- Smart Money Concepts；
- Lightweight TA-Lib / Technical indicators；
- Indicator search 简化示例；
- Signal detection；
- Pivot detection；
- Renko chart；
- Rolling OLS；
- TA-Lib time frames。

每个小节都遵循同一个结构：输入列是什么、qust 表达式怎么处理、输出列怎么读、图上应该看什么。


## 阅读方式

内容按“输入 -> qust 表达式 -> 输出/图形”的方式组织，不是零散代码片段。

建议按这个顺序读：

1. 先看输入表需要哪些列，以及这些列在策略/指标里代表什么；
2. 再看 qust 表达式每一步新增、过滤、聚合了什么；
3. 最后看 `DataFrame` 输出或 qust monitor 的真实交互输出；
4. 如果要换成自己的数据，先保证列名、类型、时间粒度一致，再改参数。

图形示例使用 qust/monitor 的原生输出，运行对应 cell 即可看到交互图。

当前主题：`indicator examples`。


In [1]:
import qust as qs
import qust.expr.vbt
import qust.expr.smc
import qust.vbt
import qust.smc
from qust import col
from qust import datasource as qds
from qust._polars import pl
from qust.monitor import mark_shape
from IPython.display import display

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(16)

DATA_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
ROWS = 6000

raw = pl.read_parquet(DATA_PATH).with_columns(pl.lit(True).alias("is_finished")).sort(["ticker", "ct", "datetime"])
source = qds.from_dataframe(raw, chunk_size=200_000)
raw = col.all.calc_data(source).sort(["ticker", "ct", "datetime"])
PLOT_ROW = raw.select("ticker", "ct").unique().sort(["ticker", "ct"]).row(0, named=True)
PLOT_TICKER = PLOT_ROW["ticker"]
PLOT_CT = PLOT_ROW["ct"]
data_all = raw.filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == PLOT_CT))
data = data_all.head(ROWS)

summary = col(
    col("datetime").min().alias("start"),
    col("datetime").max().alias("end"),
    col("close").min().alias("close_min"),
    col("close").max().alias("close_max"),
    col("close").count().alias("rows"),
).calc_data(data).with_columns(pl.lit(f"{PLOT_TICKER}/{PLOT_CT}").alias("plot_contract"))
summary


start,end,close_min,close_max,rows,plot_contract
datetime[ms],datetime[ms],f64,f64,u32,str
2022-01-04 09:00:00,2022-03-29 14:55:00,7969.0,10620.0,2519,"""AP/205"""


## 1. Hurst exponent：用 rolling 窗口估计趋势/均值回复特征

`col("close").ta.hurst(...)` 计算 Hurst exponent。这里要区分两个窗口概念：

| 参数/上下文 | 含义 |
| --- | --- |
| `.rolling(300)` | qust 的外层 rolling 上下文，每一行用最近 300 根 K 线作为输入序列 |
| `max_window=100` | Hurst R/S 算法内部使用的最大子窗口，对应 `hurst.compute_Hc(..., max_window=100)` 的含义 |
| `kind="price"` | 按价格序列解释输入，不是收益率序列 |

表达式流程：

```text
close
-> ta.hurst(max_window=100, kind="price") 核心 Hurst 计算
-> rolling(300) 每行滚动估计
-> alias("hurst") 命名输出列
```

一般读法：H 接近 0.5 更像随机游走；大于 0.5 偏趋势延续；小于 0.5 偏均值回复。实际策略里不要只用 Hurst 单独下判断，最好和波动、成交量、趋势强度一起看。


In [2]:
hurst_data = col.with_cols(
    col("close")
        .ta.hurst(max_window=100, kind="price")
        .rolling(300)
        .alias("hurst")
).calc_data(data)

hurst_data.select("datetime", "close", "hurst").filter(pl.col("hurst").is_not_null()).tail(8)


datetime,close,hurst
datetime[ms],f64,f64
2022-03-29 14:20:00,9919.0,0.655004
2022-03-29 14:25:00,9935.0,0.637217
2022-03-29 14:30:01,9938.0,0.629358
2022-03-29 14:35:01,9949.0,0.632911
2022-03-29 14:40:00,9933.0,0.638691
2022-03-29 14:45:01,9930.0,0.644962
2022-03-29 14:50:00,9929.0,0.63912
2022-03-29 14:55:00,9915.0,0.65461


In [ ]:
hurst_dashboard = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("hurst_price", show_axis_label=True)
        .kline(),
    col("datetime", "hurst")
        .monitor("hurst", show_axis_label=True)
        .line(),
).monitor.make_monitor("black").monitor.add_grid([
    ["hurst_price"],
    ["hurst"],
]).runtime()

hurst_dashboard.plot(hurst_data, open_in_jupyter=True, auto_open=False, height=760)


## 2. Smart Money Concepts：结构类行情标注

SMC 这类指标不是单条均线，而是对 K 线结构做标注。qust Python 端新增 `smc` namespace，尽量对齐 `smart-money-concepts` 的常用接口。

常用输入列：

| 算子 | 输入列 | 输出重点 |
| --- | --- | --- |
| `fvg` | `open/high/low/close` | `FVG/Top/Bottom/MitigatedIndex` |
| `swing_highs_lows` | `open/high/low/close` | `HighLow/Level`，`1` 是摆动高点，`-1` 是摆动低点 |
| `bos_choch` | `open/high/low/close` | `BOS/CHOCH/Level/BrokenIndex` |
| `ob` | `open/high/low/close/volume` | `OB/Top/Bottom/OBVolume/Percentage` |
| `liquidity` | `open/high/low/close` | 相近高低点形成的流动性簇 |
| `previous_high_low` | `datetime/high/low/close` | 前一周期高低点 |
| `sessions` | `datetime/high/low` | 指定交易时段内的活跃区间 |
| `retracements` | `open/high/low/close` | 回撤方向和幅度 |

多品种数据要接 `.over("ticker", "ct")`，否则不同品种的结构会混在一起。这里为了图更清楚，前面已经只筛了单个 `TICKER`。


In [3]:
smc_fvg = col.with_cols(
    col("open", "high", "low", "close").smc.fvg(join_consecutive=True)
).calc_data(data)

smc_swings = col.with_cols(
    col("open", "high", "low", "close").smc.swing_highs_lows(swing_length=8)
).calc_data(data)

smc_bos = col.with_cols(
    col("open", "high", "low", "close").smc.bos_choch(swing_length=8, close_break=True)
).calc_data(data)

smc_ob = col.with_cols(
    col("open", "high", "low", "close", "volume").smc.ob(swing_length=8, close_break=True)
).calc_data(data)

smc_liq = col.with_cols(
    col("open", "high", "low", "close").smc.liquidity(swing_length=8, range_percent=0.01)
).calc_data(data)

smc_phl = col.with_cols(
    col("datetime", "high", "low", "close").smc.previous_high_low("1D")
).calc_data(data)

smc_session = col.with_cols(
    col("datetime", "high", "low").smc.sessions("Custom", start_time="09:00", end_time="15:00")
).calc_data(data)

smc_retr = col.with_cols(
    col("open", "high", "low", "close").smc.retracements(swing_length=8)
).calc_data(data)

smc_counts = pl.DataFrame({
    "indicator": ["FVG", "Swing", "BOS/CHOCH", "OB", "Liquidity", "PrevHighLow rows", "Session rows", "Retracement rows"],
    "count": [
        smc_fvg.filter(pl.col("FVG") != 0).height,
        smc_swings.filter(pl.col("HighLow") != 0).height,
        smc_bos.filter((pl.col("BOS") != 0) | (pl.col("CHOCH") != 0)).height,
        smc_ob.filter(pl.col("OB") != 0).height,
        smc_liq.filter(pl.col("Liquidity") != 0).height,
        smc_phl.filter(pl.col("PreviousHigh").is_not_null()).height,
        smc_session.filter(pl.col("Active") == 1).height,
        smc_retr.filter(pl.col("Direction") != 0).height,
    ],
})

smc_counts


indicator,count
str,i64
"""FVG""",327
"""Swing""",159
"""BOS/CHOCH""",53
"""OB""",27
"""Liquidity""",44
"""PrevHighLow rows""",2475
"""Session rows""",2465
"""Retracement rows""",2435


In [4]:
display(smc_fvg.select("datetime", "FVG", "Top", "Bottom", "MitigatedIndex").filter(pl.col("FVG") != 0).head(8))
display(smc_bos.select("datetime", "BOS", "CHOCH", "Level", "BrokenIndex").filter((pl.col("BOS") != 0) | (pl.col("CHOCH") != 0)).head(8))
display(smc_ob.select("datetime", "OB", "Top", "Bottom", "OBVolume", "Percentage").filter(pl.col("OB") != 0).head(8))


datetime,FVG,Top,Bottom,MitigatedIndex
datetime[ms],i32,f64,f64,i64
2022-01-04 09:20:00,-1,8392.0,8267.0,6
2022-01-04 09:35:00,1,8373.0,8267.0,10
2022-01-04 09:55:00,-1,8383.0,8346.0,13
2022-01-04 10:30:00,-1,8331.0,8311.0,136
2022-01-04 10:45:00,-1,8259.0,8256.0,20
2022-01-04 13:35:00,-1,8235.0,8234.0,31
2022-01-04 14:30:00,1,8218.0,8210.0,41
2022-01-05 10:35:00,-1,8104.0,8040.0,63


datetime,BOS,CHOCH,Level,BrokenIndex
datetime[ms],i32,i32,f64,i64
2022-01-06 14:00:00,1,0,8178.0,113
2022-01-11 09:05:00,0,-1,8597.0,209
2022-01-12 09:10:00,-1,0,8536.0,255
2022-01-12 14:10:00,-1,0,8451.0,293
2022-01-13 14:50:00,-1,0,8408.0,343
2022-01-14 09:10:00,0,1,8492.0,346
2022-01-14 14:40:00,0,-1,8544.0,380
2022-01-19 09:40:00,-1,0,8496.0,487


datetime,OB,Top,Bottom,OBVolume,Percentage
datetime[ms],i32,f64,f64,f64,f64
2022-01-06 13:35:00,1,8148.0,8124.0,7833.0,42.347305
2022-01-12 08:59:00,-1,8592.0,8549.0,10079.0,10.066372
2022-01-12 13:40:01,-1,8483.0,8459.0,8701.0,55.044593
2022-01-13 14:30:00,-1,8455.0,8442.0,3141.0,53.719008
2022-01-19 09:35:00,-1,8511.0,8492.0,8508.0,51.168286
2022-01-19 14:55:00,1,8625.0,8612.0,4338.0,73.30402
2022-01-24 14:55:00,-1,8962.0,8914.0,7049.0,26.552598
2022-01-26 14:25:00,1,8859.0,8844.0,5020.0,20.882353


In [ ]:
smc_swing_dashboard = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("smc_price", show_axis_label=True)
        .kline(),
    col("datetime", "Level", (col("HighLow") == col.lit(1)).alias("swing_high"))
        .monitor("smc_price")
        .mark(shape=mark_shape.triangle_down, color="#ff6b6b", width=0.35),
    col("datetime", "Level", (col("HighLow") == col.lit(-1)).alias("swing_low"))
        .monitor("smc_price")
        .mark(shape=mark_shape.triangle_up, color="#4dd0e1", width=0.35),
).monitor.make_monitor("black").monitor.add_grid([["smc_price"]]).runtime()

smc_swing_dashboard.plot(smc_swings, open_in_jupyter=True, auto_open=False, height=620)


## 3. Lightweight TA-Lib / Technical indicators：常见指标组合

`ta` namespace 提供常见技术指标。这个小节一次性计算几类最常用指标：

| 指标 | 用途 |
| --- | --- |
| `sma/ema` | 趋势平滑和均线系统 |
| `rsi` | 价格强弱与超买超卖参考 |
| `bbands` | 价格相对波动带的位置 |
| `macd` | 趋势动量和信号线 |
| `atr` | 波动幅度 |
| `adx` | 趋势强度 |

状态类指标如果要逐行输出，需要接 `.expanding()` 或 `.rolling(...)`。否则很多行算子在 `calc_data` 下会输出最终状态，而不是每一行的历史状态。

表达式流程：

```text
close/high/low
-> ta 指标 helper
-> expanding() 逐行维护状态
-> with_cols(...) 回写到原表
-> monitor 多图展示
```


In [5]:
ta_data = col.with_cols(
    col("close").ta.sma(20).expanding().alias("sma20"),
    col("close").ta.ema(60).expanding().alias("ema60"),
    col("close").ta.rsi(14).expanding().alias("rsi14"),
    col("close").ta.bbands(20).expanding(),
    col("close").ta.macd(12, 26, 9).expanding(),
    col("high", "low", "close").ta.atr(14).expanding().alias("atr14"),
    col("high", "low", "close").ta.adx(14).expanding().alias("adx14"),
).calc_data(data)

ta_data.select("datetime", "close", "sma20", "ema60", "rsi14", "macd", "macdsignal", "atr14", "adx14").tail(8)


datetime,close,sma20,ema60,rsi14,macd,macdsignal,atr14,adx14
datetime[ms],f64,f64,f64,f64,f64,f64,f64,f64
2022-03-29 14:20:00,9919.0,9951.55,9903.226463,46.916766,11.275776,18.043762,30.094101,24.285853
2022-03-29 14:25:00,9935.0,9953.3,9904.268219,51.791962,9.957977,16.426605,30.015951,23.768841
2022-03-29 14:30:01,9938.0,9953.05,9905.374179,52.66969,9.051348,14.951553,29.229097,23.323671
2022-03-29 14:35:01,9949.0,9951.1,9906.804534,55.844252,9.115369,13.784316,28.998447,22.602388
2022-03-29 14:40:00,9933.0,9948.65,9907.663401,50.534829,7.785296,12.584512,29.212844,21.625327
2022-03-29 14:45:01,9930.0,9947.15,9908.395749,49.582939,6.415179,11.350646,27.911927,20.718056
2022-03-29 14:50:00,9929.0,9945.55,9909.071298,49.249923,5.188846,10.118286,27.489646,19.640431
2022-03-29 14:55:00,9915.0,9943.1,9909.265682,44.721357,3.052102,8.705049,26.954671,18.389829


In [ ]:
ta_dashboard = col(
    col("datetime", "close", "sma20", "ema60", "upperband", "middleband", "lowerband")
        .monitor("ta_price", show_axis_label=True)
        .line(),
    col("datetime", "rsi14", "adx14", "atr14")
        .monitor("ta_strength", show_axis_label=True)
        .line(),
    col("datetime", "macd", "macdsignal", "macdhist")
        .monitor("ta_macd", show_axis_label=True)
        .line(),
).monitor.make_monitor("black").monitor.add_grid([
    ["ta_price"],
    ["ta_strength"],
    ["ta_macd"],
]).runtime()

ta_dashboard.plot(ta_data, open_in_jupyter=True, auto_open=False, height=860)


## 4. Indicator search：先构造候选，再做简单评分

这里不是完整自动因子挖掘系统，而是演示 indicator search 的基本形态：

1. 先在 qust 里生成多个候选指标，例如 RSI、EMA spread、Hurst；
2. 构造一个很简单的目标变量 `ret1`，表示下一步要解释的收益；
3. 用候选指标滞后一行后和 `ret1` 相乘再取均值，得到一个粗略暴露度分数；
4. 根据分数决定哪些指标值得进一步参数优化或组合。

真实研究里可以把候选表达式放进 `pms(...)`，或者把参数范围交给 `opt_params/optuna_params` 做系统搜索。


In [6]:
feature_data = col.with_cols(
    col("close").ta.rsi(14).expanding().alias("rsi14"),
    col("close").ta.ema(20).expanding().alias("ema20"),
    col("close").ta.ema(60).expanding().alias("ema60"),
    col("close").ta.hurst(max_window=100, kind="price").rolling(300).alias("hurst300"),
    (col("close") / col("close").shift(1).expanding() - col.lit(1.0)).alias("ret1"),
).calc_data(data)

indicator_scores = col(
    (col("rsi14").shift(1).expanding() * col("ret1")).mean().alias("rsi14_x_ret"),
    ((col("ema20") - col("ema60")).shift(1).expanding() * col("ret1")).mean().alias("ema_spread_x_ret"),
    (col("hurst300").shift(1).expanding() * col("ret1")).mean().alias("hurst_x_ret"),
).calc_data(feature_data)

indicator_scores


rsi14_x_ret,ema_spread_x_ret,hurst_x_ret
f64,f64,f64
0.004046,0.001367,0.000033


In [ ]:
indicator_search_plot = col(
    col("datetime", "rsi14", "hurst300")
        .monitor("indicator_search", show_axis_label=True)
        .line(),
).monitor.make_monitor("black").monitor.add_grid([["indicator_search"]]).runtime()

indicator_search_plot.plot(feature_data, open_in_jupyter=True, auto_open=False, height=520)


## 5. Signal detection：把连续指标变成事件信号

很多策略不是直接用指标值，而是用“事件”：均线上穿、下穿、z-score 突破阈值等。

本节用 `vbt` namespace 的两个轻量入口：

| 算子 | 输入 | 输出 |
| --- | --- | --- |
| `col("fast", "slow").vbt.signal_cross()` | 两列连续指标 | `cross_up/cross_down` |
| `col("value").vbt.signal_zscore(window, threshold)` | 单列连续值 | `zscore/signal/signal_up/signal_down` |

表达式流程：

```text
close
-> EMA(20), EMA(60)
-> signal_cross() 生成均线上下穿
-> signal_zscore() 生成异常偏离信号
-> monitor 在价格和 zscore 面板上标记事件
```


In [7]:
signal_data = col.with_cols(
    col(
        col("close").ta.ema(20).expanding().alias("fast"),
        col("close").ta.ema(60).expanding().alias("slow"),
    ).vbt.signal_cross(),
    col("close").vbt.signal_zscore(window=80, threshold=2.0),
).calc_data(data)

signal_summary = col(
    col("cross_up").sum().alias("cross_up"),
    col("cross_down").sum().alias("cross_down"),
    col("signal_up").sum().alias("zscore_up"),
    col("signal_down").sum().alias("zscore_down"),
).calc_data(signal_data)

signal_summary


cross_up,cross_down,zscore_up,zscore_down
f64,f64,f64,f64
22.0,22.0,201.0,98.0


In [ ]:
signal_dashboard = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("signal_price", show_axis_label=True)
        .kline(),
    col("datetime", "close", "cross_up")
        .monitor("signal_price")
        .mark(shape=mark_shape.triangle_up, color="#4dd0e1", width=0.35),
    col("datetime", "close", "cross_down")
        .monitor("signal_price")
        .mark(shape=mark_shape.triangle_down, color="#ff6b6b", width=0.35),
    col("datetime", "zscore", "signal_up")
        .monitor("zscore")
        .mark(shape=mark_shape.triangle_up, color="#f2c94c", width=0.35),
    col("datetime", "zscore")
        .monitor("zscore", show_axis_label=True)
        .line(),
).monitor.make_monitor("black").monitor.add_grid([
    ["signal_price"],
    ["zscore"],
]).runtime()

signal_dashboard.plot(signal_data, open_in_jupyter=True, auto_open=False, height=760)


## 6. Pivot detection：标注局部高低点

`vbt.pivots(left, right)` 用左右窗口判断 pivot。它可以接一列价格，也可以接 `high, low` 两列。

参数含义：

| 参数 | 含义 |
| --- | --- |
| `left` | 左侧要比较的历史 bar 数 |
| `right` | 右侧要等待的未来 bar 数 |

因为 `right` 需要未来数据，所以 pivot detection 更适合研究、复盘、标注图形；它不是无延迟实盘信号。输出里常用列是 `pivot/pivot_high/pivot_low/level`，其中 `level` 是被标注的价格水平。


In [8]:
pivot_data = col.with_cols(
    col("high", "low").vbt.pivots(left=8, right=8)
).calc_data(data)

pivot_data.select("datetime", "pivot", "pivot_high", "pivot_low", "level").filter(pl.col("pivot") != 0).head(12)


datetime,pivot,pivot_high,pivot_low,level
datetime[ms],i32,bool,bool,f64
2022-01-04 14:30:00,1,true,false,8248.0
2022-01-05 10:45:00,-1,false,true,8022.0
2022-01-05 14:05:00,-1,false,true,7954.0
2022-01-06 11:10:00,1,true,false,8178.0
2022-01-06 13:35:00,-1,false,true,8124.0
2022-01-06 14:00:00,1,true,false,8224.0
2022-01-07 09:05:00,1,true,false,8834.0
2022-01-07 10:05:00,1,true,false,8568.0
2022-01-07 10:50:00,-1,false,true,8496.0


In [ ]:
pivot_dashboard = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("pivot_price", show_axis_label=True)
        .kline(),
    col("datetime", "level", "pivot_high")
        .monitor("pivot_price")
        .mark(shape=mark_shape.triangle_down, color="#ff6b6b", width=0.35),
    col("datetime", "level", "pivot_low")
        .monitor("pivot_price")
        .mark(shape=mark_shape.triangle_up, color="#4dd0e1", width=0.35),
).monitor.make_monitor("black").monitor.add_grid([["pivot_price"]]).runtime()

pivot_dashboard.plot(pivot_data, open_in_jupyter=True, auto_open=False, height=620)


## 7. Renko chart：用价格砖过滤时间噪声

Renko 不按固定时间出 K，而是按固定价格距离出砖。它适合观察趋势推进，因为横轴时间被弱化，价格每移动一个 `box_size` 才确认一块砖。

qust 的 Renko 设计成行算子：

| 特性 | 说明 |
| --- | --- |
| 输出行数 | 和输入行数一致，方便继续接 `.over(...)`、`.with_cols(...)`、monitor |
| `is_finished` | 当前行是否确认了一块砖 |
| 多砖情况 | 如果一行价格跨过多块砖，只保留最后一块确认砖，保持一行输入对应一行输出 |
| 参数 | `box_size` 控制砖大小，`reversal_boxes` 控制反转需要跨过几块砖 |

下面先计算 Renko，再过滤 `is_finished=True` 的确认砖来画图。


In [9]:
renko_data = col("datetime", "open", "high", "low", "close", "volume", "is_finished")    .kline.renko(box_size=1.0, reversal_boxes=1)    .calc_data(data)

renko_finished = renko_data.filter(pl.col("is_finished"))
renko_finished.head(12)


datetime,open,high,low,close,volume,is_finished
datetime[ms],f64,f64,f64,f64,f64,bool
2022-01-04 09:05:00,8379.0,8379.0,8378.0,8378.0,11169.0,true
2022-01-04 09:10:00,8303.0,8303.0,8302.0,8302.0,14001.0,true
2022-01-04 09:15:00,8281.0,8281.0,8280.0,8280.0,12839.0,true
2022-01-04 09:20:00,8247.0,8247.0,8246.0,8246.0,11496.0,true
2022-01-04 09:25:00,8266.0,8267.0,8266.0,8267.0,12307.0,true
2022-01-04 09:30:00,8322.0,8323.0,8322.0,8323.0,10889.0,true
2022-01-04 09:35:00,8370.0,8371.0,8370.0,8371.0,19223.0,true
2022-01-04 09:40:00,8384.0,8385.0,8384.0,8385.0,6610.0,true
2022-01-04 09:45:00,8385.0,8385.0,8384.0,8384.0,6471.0,true


In [ ]:
renko_dashboard = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("renko", show_axis_label=True)
        .kline(),
).monitor.make_monitor("black").monitor.add_grid([["renko"]]).runtime()

renko_dashboard.plot(renko_finished, open_in_jupyter=True, auto_open=False, height=620)


## 8. Rolling OLS：滚动估计指标暴露

`stock.ols(...)` 用来做滚动线性回归。本例构造：

- `ret`：1 bar 收益；
- `vol_dev`：当前成交量相对 50 窗口均值的偏离；
- `vol_dev_beta`：在 120 窗口内，`vol_dev` 对 `ret` 的滚动 beta。

表达式流程：

```text
close, volume
-> ret 和 vol_dev
-> stock.ols(stats_method="beta").rolling(120)
-> 输出 beta 曲线
```

这个模式可以扩展到因子暴露、风险暴露、价量关系检验等场景。


In [10]:
ols_data = col.with_cols(
    (col("close") / col("close").shift(1).expanding() - col.lit(1.0)).alias("ret"),
    (col("volume") - col("volume").mean().rolling(50)).alias("vol_dev"),
).with_cols(
    col("ret", "vol_dev").stock.ols(stats_method="beta").rolling(120)
).calc_data(data)

ols_data.select("datetime", "ret", "vol_dev", "vol_dev_beta").tail(10)


datetime,ret,vol_dev,vol_dev_beta
datetime[ms],f64,f64,f64
2022-03-29 14:10:01,0.000603,-1005.42,-3.9655e-7
2022-03-29 14:15:00,-0.002411,-731.3,-3.9398e-7
2022-03-29 14:20:00,-0.001108,-440.48,-3.9299e-7
2022-03-29 14:25:00,0.001613,-610.5,-3.9698e-7
2022-03-29 14:30:01,0.000302,-1105.68,-3.9400e-7
2022-03-29 14:35:01,0.001107,-863.9,-3.9247e-7
2022-03-29 14:40:00,-0.001608,-689.12,-3.9267e-7
2022-03-29 14:45:01,-0.000302,-1100.44,-3.9095e-7
2022-03-29 14:50:00,-0.000101,-455.28,-3.9146e-7


In [ ]:
ols_dashboard = col(
    col("datetime", "vol_dev_beta")
        .monitor("rolling_ols", show_axis_label=True)
        .line(),
).monitor.make_monitor("black").monitor.add_grid([["rolling_ols"]]).runtime()

ols_dashboard.plot(ols_data, open_in_jupyter=True, auto_open=False, height=520)


## 9. TA-Lib time frames：先升周期，再做指标

多周期指标不要手动在 Python 里拼循环。qust 的做法是：

```text
基础 K 线
-> kline.rl5m / kline.rl30m 合成更高周期 K 线
-> filter(is_finished) 只保留确认 K
-> 在高周期 K 上继续计算 RSI/EMA/其它 TA
-> monitor 展示高周期价格和指标
```

这里用 `.expanding()` 展示逐行合成过程。`is_finished=True` 表示这根高周期 K 已经收完，适合拿去做指标和信号。


In [11]:
tf5 = col("datetime", "open", "high", "low", "close", "volume").kline.rl5m.expanding().calc_data(data)
tf30 = col("datetime", "open", "high", "low", "close", "volume").kline.rl30m.expanding().calc_data(data)

tf5_finished = tf5.filter(pl.col("is_finished"))
tf30_finished = tf30.filter(pl.col("is_finished"))

tf5_ta = col.with_cols(
    col("close").ta.rsi(14).expanding().alias("rsi14_5m"),
    col("close").ta.ema(20).expanding().alias("ema20_5m"),
).calc_data(tf5_finished)

pl.DataFrame({
    "frame": ["5m", "30m"],
    "finished_rows": [tf5_finished.height, tf30_finished.height],
})


frame,finished_rows
str,i64
"""5m""",1260
"""30m""",447


In [ ]:
timeframe_dashboard = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("tf5_price", show_axis_label=True)
        .kline(),
    col("datetime", "rsi14_5m", "ema20_5m")
        .monitor("tf5_ta", show_axis_label=True)
        .line(),
).monitor.make_monitor("black").monitor.add_grid([
    ["tf5_price"],
    ["tf5_ta"],
]).runtime()

timeframe_dashboard.plot(tf5_ta, open_in_jupyter=True, auto_open=False, height=760)
